# الفصل 34: DPO وKTO وORPO: ما بعد RLHF

> RLHF/PPO (الفصل السابق) فعّال لكنه معقّد جدًا هندسيًا: 4 نماذج، عدم استقرار تدريبي معروف، حساسية شديدة للمُفرطات (hyperparameters)، وتكلفة حوسبية عالية.

- **الجزء:** 7 — التخصيص الفعّال والمحاذاة
- **الفصل:** 34
- **المستوى:** خبير
- **زمن القراءة:** 20 دقيقة
- **المتطلبات:** الفصل 33: RLHF وPPO بعمق
- **الوسوم:** DPO، KTO، ORPO، IPO، Preference Tuning

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch34-dpo-kto-orpo.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-07-peft-alignment%2Fch34-dpo-kto-orpo.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-07-peft-alignment/ch34-dpo-kto-orpo.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch34-dpo-kto-orpo.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch34-dpo-kto-orpo.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch34-dpo-kto-orpo.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch34-dpo-kto-orpo.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-07-peft-alignment%2Fch34-dpo-kto-orpo.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-07-peft-alignment/ch34-dpo-kto-orpo.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch34-dpo-kto-orpo.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch34-dpo-kto-orpo.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch34-dpo-kto-orpo.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> RLHF/PPO (الفصل السابق) فعّال لكنه معقّد جدًا هندسيًا: 4 نماذج، عدم استقرار تدريبي معروف، حساسية شديدة للمُفرطات (hyperparameters)، وتكلفة حوسبية عالية.

> **بيانات الفصل**
>
> - **الجزء:** 7 — التخصيص الفعّال والمحاذاة
> - **المستوى:** خبير
> - **زمن القراءة:** 20 دقيقة
> - **المتطلبات:** الفصل 33: RLHF وPPO بعمق


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم IPO — إصلاح نقطة ضعف DPO
- فهم KTO — المحاذاة بدون أزواج تفضيل!
- فهم ORPO — دمج SFT والمحاذاة في خطوة واحدة


## 1. لماذا يهمّك هذا الموضوع؟

RLHF/PPO (الفصل السابق) فعّال لكنه **معقّد جدًا هندسيًا**: 4 نماذج، عدم استقرار تدريبي معروف، حساسية شديدة للمُفرطات (hyperparameters)، وتكلفة حوسبية عالية. في 2023، ورقة **DPO (Direct Preference Optimization)** أحدثت ثورة بإثبات أنه يمكن تحقيق **نفس هدف RLHF رياضيًا** دون الحاجة لنموذج مكافأة منفصل، ولا لحلقة RL كاملة، بل فقط عبر **دالة خسارة واحدة بسيطة شبيهة بـ Supervised Learning**. اليوم، معظم النماذج مفتوحة المصدر الحديثة (Llama 3، Zephyr، Qwen، Mistral المُحاذاة) تستخدم DPO أو مشتقاتها بدلًا من PPO الكلاسيكي.


## 2. الخريطة الذهنية


```text
تطوّر تقنيات المحاذاة بعد SFT:

RLHF/PPO (2022)          →    معقّد، 4 نماذج، RL كامل
       │
       ▼
DPO (2023)                →    نموذجان فقط (policy + reference)
       │                       لا حاجة لـ RM منفصل ولا RL فعلي!
       ├─→ IPO (2023)     →    نسخة مُعدَّلة رياضيًا، أكثر مقاومة للـ overfitting
       ├─→ KTO (2024)     →    لا يحتاج أزواج تفضيل، فقط تصنيف ثنائي (جيد/سيء)
       └─→ ORPO (2024)    →    يدمج SFT والمحاذاة في خطوة تدريب واحدة فقط!
```


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

الفكرة العبقرية في DPO: بدلًا من "تدريب نموذج مكافأة، ثم استخدام RL لتحسين السياسة بناءً عليه" (خطوتان منفصلتان ومعقّدتان)، يمكن **حل المعادلات رياضيًا مُسبقًا** لنُظهِر أن "نموذج المكافأة الضمني الأمثل" يمكن التعبير عنه مباشرة كدالة في **السياسة نفسها** (policy). بمعنى آخر: **السياسة المُحسَّنة هي نفسها نموذج المكافأة الضمني**! هذا يُلغي الحاجة الكاملة لتدريب RM منفصل أو تشغيل حلقة RL، ونحصل بدلًا من ذلك على **دالة خسارة مباشرة (closed-form)** نُحسِّنها بـ Gradient Descent العادي تمامًا كأي مهمة Supervised Learning.

### 🟡 طبقة الآلية — الاشتقاق الرياضي لـ DPO

نبدأ من هدف RLHF القياسي (تعظيم المكافأة مع قيد KL):


```text
max_π  E_{y~π(y|x)}[r(x,y)] - β·KL(π(y|x) || π_ref(y|x))
```


الحل التحليلي المعروف لهذه المسألة (من نظرية RL، بصيغة مغلقة) هو:


```text
π*(y|x) = (1/Z(x)) · π_ref(y|x) · exp(r(x,y)/β)
```


حيث `Z(x)` هو ثابت تطبيع (partition function). بإعادة ترتيب هذه المعادلة لحل `r(x,y)` بدلالة `π*`:


```text
r(x,y) = β·log(π*(y|x)/π_ref(y|x)) + β·log Z(x)
```


الآن، نُعوِّض هذه الصيغة في دالة خسارة Bradley-Terry الأصلية لـ RM (من الفصل 33). الجميل هنا أن ثابت `log Z(x)` **يُحذَف تلقائيًا** لأنه يظهر في كل من `r(x, y_w)` و`r(x, y_l)` وبالتالي يُلغى عند طرحهما:


```text
L_DPO(θ) = -E[log σ(β·log(π_θ(y_w|x)/π_ref(y_w|x)) - β·log(π_θ(y_l|x)/π_ref(y_l|x)))]
```


هذه هي **دالة خسارة DPO الكاملة** — وهي ببساطة: حساب log-probability نسبي (مقارنة بـ reference) لكل من الاستجابة المُفضَّلة والمرفوضة، ثم تطبيق نفس منطق Bradley-Terry.

### 🔴 طبقة التعمّق — تطبيق DPO من الصفر


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def compute_sequence_logprobs(model, input_ids: torch.Tensor,
                                attention_mask: torch.Tensor,
                                labels: torch.Tensor) -> torch.Tensor:
    """
    حساب مجموع log-probabilities لتسلسل كامل (الاستجابة فقط،
    وليس التعليمة/prompt — يجب إخفاء التعليمة بـ labels=-100).
    هذا الـ building block مُستخدَم في كل من DPO/KTO/ORPO.
    """
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]   # إزاحة بمقدار واحد (next-token prediction)
    labels = labels[:, 1:]

    log_probs = F.log_softmax(logits, dim=-1)

    # إخفاء مواضع الـ prompt/padding (labels == -100) من حساب الخسارة
    mask = (labels != -100)
    labels_clamped = labels.clamp(min=0)  # لتفادي index error مؤقتًا
    token_logprobs = torch.gather(
        log_probs, dim=2, index=labels_clamped.unsqueeze(-1)
    ).squeeze(-1)

    token_logprobs = token_logprobs * mask  # تصفير المواضع المُخفاة
    sequence_logprobs = token_logprobs.sum(dim=1)  # مجموع log-probs للاستجابة كاملة
    return sequence_logprobs

class DPOTrainer:
    """
    تطبيق كامل لـ Direct Preference Optimization (Rafailov et al., 2023).
    يتطلب فقط نموذجين: Policy (قابل للتدريب) وReference (مُجمَّد).
    """
    def __init__(self, policy_model: nn.Module, reference_model: nn.Module,
                 beta: float = 0.1):
        self.policy = policy_model
        self.reference = reference_model
        self.beta = beta

        for p in self.reference.parameters():
            p.requires_grad = False
        self.reference.eval()

    def compute_dpo_loss(self, batch: dict) -> dict:
        """
        batch يحتوي على: chosen_input_ids, chosen_attention_mask, chosen_labels,
                          rejected_input_ids, rejected_attention_mask, rejected_labels
        """
        # Log-probs من السياسة الحالية (قابلة للتدريب)
        policy_chosen_logps = compute_sequence_logprobs(
            self.policy, batch["chosen_input_ids"],
            batch["chosen_attention_mask"], batch["chosen_labels"]
        )
        policy_rejected_logps = compute_sequence_logprobs(
            self.policy, batch["rejected_input_ids"],
            batch["rejected_attention_mask"], batch["rejected_labels"]
        )

        # Log-probs من النموذج المرجعي (بدون تدرجات)
        with torch.no_grad():
            ref_chosen_logps = compute_sequence_logprobs(
                self.reference, batch["chosen_input_ids"],
                batch["chosen_attention_mask"], batch["chosen_labels"]
            )
            ref_rejected_logps = compute_sequence_logprobs(
                self.reference, batch["rejected_input_ids"],
                batch["rejected_attention_mask"], batch["rejected_labels"]
            )

        # النسب اللوغاريتمية (implicit reward) لكل من المُفضَّل والمرفوض
        chosen_log_ratio = policy_chosen_logps - ref_chosen_logps
        rejected_log_ratio = policy_rejected_logps - ref_rejected_logps

        # دالة خسارة DPO النهائية (معادلة الاشتقاق أعلاه)
        logits = self.beta * (chosen_log_ratio - rejected_log_ratio)
        loss = -F.logsigmoid(logits).mean()

        # مقاييس مفيدة للمراقبة أثناء التدريب
        with torch.no_grad():
            implicit_reward_chosen = self.beta * chosen_log_ratio
            implicit_reward_rejected = self.beta * rejected_log_ratio
            accuracy = (implicit_reward_chosen > implicit_reward_rejected).float().mean()
            reward_margin = (implicit_reward_chosen - implicit_reward_rejected).mean()

        return {
            "loss": loss,
            "accuracy": accuracy.item(),
            "reward_margin": reward_margin.item(),
            "chosen_reward": implicit_reward_chosen.mean().item(),
            "rejected_reward": implicit_reward_rejected.mean().item(),
        }

    def training_step(self, batch: dict, optimizer):
        optimizer.zero_grad()
        metrics = self.compute_dpo_loss(batch)
        metrics["loss"].backward()
        optimizer.step()
        return {k: (v if isinstance(v, float) else v.item())
                for k, v in metrics.items()}


### IPO — إصلاح نقطة ضعف DPO

DPO يُعاني من مشكلة: دالة `logsigmoid` تدفع الفارق `chosen_log_ratio - rejected_log_ratio` إلى **اللانهاية** نظريًا (لا يوجد حد أقصى يُرضي الخسارة)، مما قد يُسبب Overfitting شديدًا خصوصًا إذا كانت بيانات التفضيل تحتوي أزواجًا "سهلة جدًا" يُميّزها النموذج بسرعة.


In [ ]:
def ipo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta: float = 0.1):
    """
    IPO (Azar et al., 2023): بدلًا من logsigmoid غير المحدودة،
    نستخدم خسارة تربيعية (MSE-style) مع هدف ثابت = 1/(2β).
    هذا يمنع "الثقة المفرطة" (overconfidence) ويُحسِّن التعميم،
    خصوصًا عندما تحتوي البيانات على تفضيلات قوية جدًا أو حتمية.
    """
    chosen_log_ratio = policy_chosen_logps - ref_chosen_logps
    rejected_log_ratio = policy_rejected_logps - ref_rejected_logps

    h = chosen_log_ratio - rejected_log_ratio
    # الهدف: h يجب أن يقترب من 1/(2β) بالضبط، وليس "أكبر ما يمكن"
    loss = (h - 1 / (2 * beta)) ** 2
    return loss.mean()


### KTO — المحاذاة بدون أزواج تفضيل!

تحدٍ عملي كبير في DPO: **تحتاج أزواج مُقارَنة صريحة** (chosen/rejected لنفس الـ prompt)، وهذا يتطلب جمع بيانات أكثر تعقيدًا. KTO (Kahneman-Tversky Optimization) مُستلهَمة من **نظرية الاحتمال المرجّح البشري** (Prospect Theory لنظرية Kahneman-Tversky في الاقتصاد السلوكي)، وتحتاج فقط **تصنيفًا ثنائيًا مستقلًا**: هل هذه الاستجابة "مرغوبة" أم "غير مرغوبة"؟ — بدون الحاجة لمقارنتها باستجابة أخرى لنفس الـ prompt.


In [ ]:
def kto_loss(policy_logps: torch.Tensor, ref_logps: torch.Tensor,
             is_desirable: torch.Tensor, beta: float = 0.1,
             desirable_weight: float = 1.0, undesirable_weight: float = 1.0):
    """
    KTO (Ethayarajh et al., 2024): لا يحتاج أزواج (chosen, rejected)!
    كل عينة بيانات مستقلة، فقط بعلامة ثنائية: مرغوبة (1) أو غير مرغوبة (0).

    مُستوحى من نظرية "النفور من الخسارة" (loss aversion) البشرية:
    البشر يتفاعلون بشكل غير متماثل مع المكاسب والخسائر، لذا KTO
    تستخدم دالة قيمة (value function) غير خطية وغير متماثلة.
    """
    log_ratio = policy_logps - ref_logps
    kl_estimate = log_ratio.mean().detach()  # تقدير KL على مستوى الدفعة

    # دالة القيمة غير المتماثلة (مستوحاة من Prospect Theory)
    desirable_losses = 1 - torch.sigmoid(
        beta * (log_ratio - kl_estimate)
    )
    undesirable_losses = 1 - torch.sigmoid(
        beta * (kl_estimate - log_ratio)
    )

    losses = torch.where(
        is_desirable.bool(),
        desirable_weight * desirable_losses,
        undesirable_weight * undesirable_losses,
    )
    return losses.mean()


> 💡 **الميزة العملية الكبرى لـ KTO**: يمكن استخدام بيانات موجودة بالفعل في معظم المنتجات (مثل 👍/👎 على استجابات chatbot حقيقي) مباشرة دون الحاجة لإعادة جمع بيانات مُقارَنة زوجية مُكلفة.

### ORPO — دمج SFT والمحاذاة في خطوة واحدة

ORPO (Odds Ratio Preference Optimization) يذهب خطوة أبعد: **يُلغي الحاجة لمرحلة SFT منفصلة بالكامل**، ويُدرِّب النموذج من الأساسي مباشرة بخسارة واحدة تجمع (1) تعلّم اتباع التعليمات و(2) تفضيل الاستجابة الجيدة على السيئة، في آن واحد.


In [ ]:
def orpo_loss(model, chosen_batch: dict, rejected_batch: dict,
              lambda_weight: float = 0.1):
    """
    ORPO (Hong et al., 2024): خسارة واحدة مُركَّبة من جزأين:

    1. SFT Loss (Cross-Entropy) على الاستجابة المُفضَّلة فقط
       (نفس هدف التدريب الخاضع للإشراف العادي)

    2. Odds Ratio Loss: تعظيم نسبة الأرجحية (odds) للاستجابة
       المُفضَّلة على المرفوضة، دون الحاجة لأي reference model!
       (الفارق الجوهري عن DPO: لا نموذج مرجعي مُجمَّد مطلوب)
    """
    chosen_outputs = model(input_ids=chosen_batch["input_ids"],
                             attention_mask=chosen_batch["attention_mask"])
    rejected_outputs = model(input_ids=rejected_batch["input_ids"],
                               attention_mask=rejected_batch["attention_mask"])

    # الجزء الأول: SFT Loss قياسية على الاستجابة المُفضَّلة
    sft_loss = F.cross_entropy(
        chosen_outputs.logits[:, :-1].reshape(-1, chosen_outputs.logits.size(-1)),
        chosen_batch["labels"][:, 1:].reshape(-1),
        ignore_index=-100,
    )

    # الجزء الثاني: Odds Ratio بين الاستجابتين
    def compute_avg_logprob(logits, labels):
        log_probs = F.log_softmax(logits[:, :-1], dim=-1)
        labels_shifted = labels[:, 1:]
        mask = (labels_shifted != -100)
        labels_clamped = labels_shifted.clamp(min=0)
        token_logps = torch.gather(
            log_probs, 2, labels_clamped.unsqueeze(-1)
        ).squeeze(-1) * mask
        # متوسط (وليس مجموع) لتطبيع طول التسلسل - مهم لصيغة Odds Ratio
        return token_logps.sum(dim=1) / mask.sum(dim=1).clamp(min=1)

    chosen_logp = compute_avg_logprob(chosen_outputs.logits, chosen_batch["labels"])
    rejected_logp = compute_avg_logprob(rejected_outputs.logits, rejected_batch["labels"])

    # log odds = log(p / (1-p))، حيث p = exp(avg_logprob)
    log_odds_chosen = chosen_logp - torch.log1p(-torch.exp(chosen_logp).clamp(max=1 - 1e-6))
    log_odds_rejected = rejected_logp - torch.log1p(-torch.exp(rejected_logp).clamp(max=1 - 1e-6))

    odds_ratio_loss = -F.logsigmoid(log_odds_chosen - log_odds_rejected).mean()

    total_loss = sft_loss + lambda_weight * odds_ratio_loss
    return {
        "total_loss": total_loss,
        "sft_loss": sft_loss.item(),
        "odds_ratio_loss": odds_ratio_loss.item(),
    }


---


## 4. مثال محلول خطوة بخطوة

**المسألة**: احسب خسارة DPO لعيّنة واحدة يدويًا، مع تفسير أثر β.


In [ ]:
def manual_dpo_example():
    # افتراضات مبسّطة لقيم log-probability (مجموع log-probs للاستجابة كاملة)
    policy_chosen_logp = torch.tensor(-12.5)    # النموذج الحالي "واثق نسبيًا" من المُفضَّل
    policy_rejected_logp = torch.tensor(-15.0)  # وأقل ثقة في المرفوض (جيد!)
    ref_chosen_logp = torch.tensor(-13.0)       # النموذج المرجعي (قبل أي تحديث)
    ref_rejected_logp = torch.tensor(-13.5)

    for beta in [0.05, 0.1, 0.5]:
        chosen_ratio = policy_chosen_logp - ref_chosen_logp     # 0.5
        rejected_ratio = policy_rejected_logp - ref_rejected_logp  # -1.5

        logits = beta * (chosen_ratio - rejected_ratio)  # beta * 2.0
        loss = -F.logsigmoid(logits)

        print(f"β={beta}: implicit_logits={logits.item():.3f}, "
              f"loss={loss.item():.4f}")

manual_dpo_example()
# β=0.05: implicit_logits=0.100, loss=0.6444
# β=0.1:  implicit_logits=0.200, loss=0.5981
# β=0.5:  implicit_logits=1.000, loss=0.3133


> 🔴 **ملاحظة تفسيرية**: β أعلى يعني أن النموذج "يثق أكثر" في إشارة التفضيل ويدفع الفارق بقوة أكبر (خسارة أقل هنا لأن الفارق بالفعل في الاتجاه الصحيح)، لكن β أعلى جدًا قد يُسبب انحرافًا كبيرًا جدًا عن `π_ref` إذا لم تُراقَب بعناية — تمامًا كدور `kl_coef` في PPO لكن بشكل ضمني داخل معادلة واحدة.

---


## 5. نصائح وحيل

| النصيحة | التفصيل |
|---|---|
| ابدأ بـ β=0.1 | نقطة انطلاق قياسية شائعة في معظم أوراق DPO التطبيقية |
| راقب `reward_margin` | يجب أن يزداد بثبات أثناء التدريب؛ ثباته قرب الصفر يعني عدم تعلّم فعّال |
| احذر من Overfitting في DPO | استخدم عدد epochs محدود جدًا (1-3) — DPO يميل للـ overfitting بسرعة أكبر من SFT |
| جرّب IPO إذا رأيت تدهورًا بعد عدة epochs | علامة كلاسيكية على مشكلة عدم الحدّية في DPO الأصلي |
| استخدم ORPO لتبسيط الأنبوب الكامل | يوفّر خطوة تدريب كاملة ويُقلِّل تعقيد الأنبوب الهندسي بشكل ملحوظ |


## 6. أخطاء شائعة ومصائد

- **نسيان تطبيع طول التسلسل في بعض الصيغ (كما في ORPO)**: مجموع log-probs الخام متحيّز لصالح التسلسلات الأقصر؛ المتوسط ضروري في بعض المعادلات.
- **استخدام نفس LR المُستخدَم في SFT**: DPO يحتاج عادة LR أقل بكثير (5e-7 إلى 5e-6) نظرًا لحساسيته العالية.
- **تجاهل جودة بيانات التفضيل**: DPO حساس جدًا لجودة أزواج (chosen, rejected) — بيانات صاخبة (noisy) تؤدي لتدهور سريع وواضح.
- **تطبيق DPO مباشرة على نموذج أساسي (بدون SFT مسبق)**: يؤدي لنتائج ضعيفة جدًا؛ DPO مُصمَّم كمرحلة **بعد** SFT (باستثناء ORPO الذي صُمِّم خصيصًا للعمل من الأساسي مباشرة).


## 7. جدول مقارنة

| التقنية | يحتاج Reward Model؟ | يحتاج أزواج تفضيل؟ | يحتاج Reference Model؟ | التعقيد الهندسي | الاستقرار |
|---|---|---|---|---|---|
| **PPO (RLHF)** | ✅ نعم (منفصل) | ✅ نعم (لتدريب RM) | ✅ نعم | 🔴 عالٍ جدًا (4 نماذج) | متوسط |
| **DPO** | ❌ لا (ضمني) | ✅ نعم | ✅ نعم | 🟡 متوسط (نموذجان) | جيد |
| **IPO** | ❌ لا | ✅ نعم | ✅ نعم | 🟡 متوسط | أفضل من DPO (أقل overfitting) |
| **KTO** | ❌ لا | ❌ لا (تصنيف ثنائي فقط) | ✅ نعم | 🟡 متوسط | جيد، مرن في جمع البيانات |
| **ORPO** | ❌ لا | ✅ نعم | ❌ **لا حاجة!** | 🟢 الأبسط (نموذج واحد فقط) | جيد، يدمج SFT+Align |


## 8. ملخص أفضل الممارسات

- للمشاريع الحديثة: ابدأ بـ **DPO** كخط أساس قياسي (الأكثر دعمًا في المكتبات: TRL, Axolotl).
- إذا كانت بيانات التفضيل محدودة لكن لديك تصنيفات ثنائية (👍/👎) من منتج حقيقي: استخدم **KTO**.
- إذا أردت تبسيط الأنبوب الكامل (موارد هندسية محدودة): استخدم **ORPO** لدمج SFT والمحاذاة.
- احتفظ دائمًا بتقييم بشري نهائي (الفصل 43: LLM-as-a-Judge) بغض النظر عن التقنية المُستخدَمة — لا تثق فقط بمقاييس التدريب الداخلية.
- راقب Catastrophic Forgetting (الفصل 24) بعد أي مرحلة محاذاة، تمامًا كما بعد أي ضبط دقيق آخر.


## 9. أسئلة مقابلات

**س: كيف يتجنّب DPO الحاجة لتدريب نموذج مكافأة منفصل، مع أنه يحقق نفس هدف RLHF نظريًا؟**

**ج**: من خلال استغلال الحل التحليلي المغلق (closed-form) لمسألة التحسين المُقيَّدة بـ KL في RLHF. رياضيًا، يمكن إثبات أن السياسة المثلى `π*` ترتبط بدالة المكافأة `r` بعلاقة مباشرة: `r(x,y) = β·log(π*(y|x)/π_ref(y|x)) + const`. بتعويض هذه العلاقة في خسارة Bradley-Terry الأصلية (المستخدَمة لتدريب RM)، يختفي ثابت التطبيع `Z(x)` تلقائيًا (لأنه يظهر في طرفي المقارنة ويُلغى بالطرح)، فنحصل على دالة خسارة تعتمد **فقط على السياسة ونسختها المرجعية**، دون الحاجة لتدريب `r` بشكل منفصل إطلاقًا — السياسة نفسها تُصبح "نموذج المكافأة الضمني".

**س: متى تُفضِّل KTO على DPO عمليًا؟**

**ج**: عندما لا تملك بيانات تفضيل **زوجية** مُهيكلة (أي عدة استجابات لنفس الـ prompt مُرتَّبة من مُقيِّم واحد)، بل تملك بيانات تصنيف ثنائي مستقل (كل استجابة بمفردها، مع علامة "جيدة" أو "سيئة") — وهذا شائع جدًا في بيانات الإنتاج الفعلية (مثل أزرار 👍/👎 في تطبيق محادثة حقيقي، أو بيانات شكاوى/رضا مستخدمين). KTO مبني نظريًا على افتراض عدم الحاجة للمقارنة الصريحة بين عيّنتين، فيُقلِّل بشكل كبير من تكلفة وتعقيد جمع البيانات مقارنة بـ DPO.

**س: ما السبب الرياضي الذي يجعل DPO عرضة للـ Overfitting أكثر من IPO؟**

**ج**: دالة `logsigmoid` في خسارة DPO تستمر في الانخفاض كلما ازداد الفارق `β·(chosen_log_ratio - rejected_log_ratio)` نحو اللانهاية الموجبة — بمعنى آخر، لا يوجد "هدف محدود" يُرضي الخسارة تمامًا، فتستمر عملية التحسين في دفع النموذج نحو **ثقة متطرفة** (قرب احتمال 1 للمُفضَّل وقرب صفر للمرفوض) حتى لو كانت هذه الثقة غير مبرَّرة إحصائيًا. IPO تستبدل هذا بخسارة تربيعية (MSE) مع **هدف ثابت محدد مسبقًا (1/2β)**، فبمجرد وصول الفارق لهذا الهدف، تتوقف الخسارة عن الدفع لمزيد من التطرف، مما يُحقق تنظيمًا (regularization) ضمنيًا أفضل.


## 10. تمرين عملي

1. نفّذ `DPOTrainer` كاملًا، ودرّبه على مجموعة بيانات تفضيل عربية (يمكن استخدام ترجمة جزء من `Anthropic/hh-rlhf` أو بناء 100 زوج يدويًا).
2. راقب `accuracy` و`reward_margin` عبر خطوات التدريب، وتأكد من أنهما يتحسنان باطراد.
3. كرّر نفس التجربة بخسارة IPO، وقارن استقرار النتائج بعد 5 epochs (حيث يُتوقَّع أن يبدأ DPO بالتدهور بينما يبقى IPO أكثر ثباتًا).


## 11. مراجع للتعمّق

- Rafailov et al. (2023) — *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*
- Azar et al. (2023) — *A General Theoretical Paradigm to Understand Learning from Human Preferences* (IPO)
- Ethayarajh et al. (2024) — *KTO: Model Alignment as Prospect Theoretic Optimization*
- Hong et al. (2024) — *ORPO: Monolithic Preference Optimization without Reference Model*
- Tversky & Kahneman (1992) — *Advances in Prospect Theory* (الأساس النظري لـ KTO)


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التخصيص الفعّال والمحاذاة / الفصل 34: DPO وKTO وORPO: ما بعد RLHF.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
